# Safe-Sora results for three MOSE samples

Safe-Sora graphical watermark embedding, object removal, and watermark recovery are shown for three fixed MOSE samples.

```text
sampled original -> Safe-Sora embedding -> watermarked video -> ProPainter or DiffuEraser -> graphical watermark recovery
```

Safe-Sora reconstructs a graphical watermark image. It does not produce a spatial tamper mask, so predicted-mask and IoU fields are not applicable.


## Metric guide

| Term | Meaning | Interpretation |
| --- | --- | --- |
| PSNR | Logarithmic similarity derived from mean squared error. | Higher is better. Values around 8 dB indicate weak graphical recovery. Values above 40 dB indicate small pixel differences. |
| MSE | Mean squared 8-bit pixel error. | Lower is better. Large errors receive extra weight because they are squared. |
| MAE | Mean absolute 8-bit pixel error. | Lower is better. It is the average absolute change per channel value. |
| VAE reconstruction fidelity | Difference between the selected input and its VAE reconstruction. | Separates ordinary VAE change from the additional watermark change. |
| Embedding fidelity | Difference between the VAE cover and watermarked output. | High PSNR and low errors mean the embedded watermark causes small pixel changes. |
| Recovery image | Reference watermark, recovered watermark, and amplified error shown side by side. | It is not an object mask and cannot be used for IoU. |
| Runtime | Wall-clock time recorded on the experiment hardware. | Useful for this run, not a hardware-independent benchmark. |
| SHA-256 | File fingerprint stored in the JSON record. | Confirms exact file identity. It is not a quality metric. |


## Notebook setup

After a fresh clone, run `python3 scripts/prepare_mose_notebooks.py` from the repository root. This downloads and verifies the 48 videos shared by the three method notebooks. Run the next cell once to enable the video, image, download, and metadata viewers. If Git LFS has left a pointer file in place of a video, the viewer reports the preparation command instead of displaying a broken player.


In [ ]:
import base64
from pathlib import Path
import html
import json
from IPython.display import HTML, Markdown, display

LFS_POINTER_PREFIX = b"version https://git-lfs.github.com/spec/v1"

def find_repo_root():
    for start in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (start / "evidence/mose-task1/review50/manifest.json").is_file():
            return start
    raise FileNotFoundError("Run this notebook from inside the paper-reproductions checkout.")

REPO_ROOT = find_repo_root()
REVIEW_ROOT = REPO_ROOT / "evidence/mose-task1/review50"

def artifact_data_url(source, mime_type):
    if not source.is_file():
        raise FileNotFoundError(source)
    payload = source.read_bytes()
    if payload.startswith(LFS_POINTER_PREFIX):
        raise RuntimeError(
            f"{source} is a Git LFS pointer, not the real media file. "
            "From the repository root run: python3 scripts/prepare_mose_notebooks.py"
        )
    return f"data:{mime_type};base64,{base64.b64encode(payload).decode('ascii')}"

def show_videos(items, columns=3):
    cards = []
    for title, relative, explanation in items:
        if relative is None:
            cards.append(
                f"<section class='media-card'><h4>{html.escape(title)}</h4>"
                f"<p><strong>Not applicable.</strong></p><p>{html.escape(explanation)}</p></section>"
            )
            continue
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "video/mp4")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<video controls muted loop playsinline preload='metadata' src='{data_url}'></video>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this video</a></p>"
            "</section>"
        )
    display(HTML(
        f"<style>.media-grid{{display:grid;grid-template-columns:repeat({columns},minmax(220px,1fr));gap:16px}}"
        ".media-card{border:1px solid #d0d7de;border-radius:8px;padding:12px;background:#fff;color:#24292f}"
        ".media-card video{width:100%;background:#000}.media-card h4{margin:0 0 8px}.media-card p{font-size:13px}</style>"
        f"<div class='media-grid'>{''.join(cards)}</div>"
    ))

def show_images(items, columns=2):
    cards = []
    for title, relative, explanation in items:
        source = REVIEW_ROOT / relative
        data_url = artifact_data_url(source, "image/png")
        cards.append(
            "<section class='media-card'>"
            f"<h4>{html.escape(title)}</h4>"
            f"<img style='width:100%' src='{data_url}' alt='{html.escape(title)}'>"
            f"<p>{html.escape(explanation)}</p>"
            f"<p><a href='{data_url}' download='{html.escape(source.name)}'>Download this image</a></p>"
            "</section>"
        )
    display(HTML(
        f"<div class='media-grid' style='grid-template-columns:repeat({columns},minmax(260px,1fr))'>"
        f"{''.join(cards)}</div>"
    ))

def show_metadata(paths):
    for relative in paths:
        path = REVIEW_ROOT / relative
        payload = json.loads(path.read_text())
        display(Markdown(f"#### `{relative}`"))
        print(json.dumps(payload, indent=2))

print(f"Loaded evidence from {REVIEW_ROOT}")


## 1. Sample `02b72b8d`


### Source and unwatermarked removal reference

The source has 67 frames at 960 x 480, 25 FPS, and a duration of about 2.68 seconds. The raw mask marks the object selected for removal. The two unwatermarked outputs show how ProPainter and DiffuEraser behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/02b72b8d/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/02b72b8d/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/02b72b8d/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/02b72b8d/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/02b72b8d/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/02b72b8d/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/02b72b8d/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/02b72b8d/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.')])


### Safe-Sora results

Safe-Sora selected 16 frames at indices `0, 4, 9, 13, 18, 22, 26, 31, 35, 40, 44, 48, 53, 57, 62, 66` and resized them to 512 x 320. Embedding took 5.783 s.

DiffuEraser requires at least 22 frames. The final frame and mask were repeated to 24 frames during inference, then the output was trimmed back to 16 genuine frames.

#### Embedding fidelity

| Comparison | PSNR | MSE | MAE |
| --- | ---: | ---: | ---: |
| Selected input to VAE cover | 32.0602 dB | 40.4635 | 4.1374 |
| VAE cover to watermarked output | **41.5799 dB** | 4.5194 | 1.5960 |

The first row measures ordinary VAE reconstruction change. The second row isolates the additional visible change caused by watermark embedding.

#### Removal and recovery

| Metric | ProPainter | DiffuEraser |
| --- | ---: | ---: |
| Removal runtime | 8.048 s | 27.696 s |
| Reported output frames | 16 | 16 |
| Recovery PSNR | **8.2317 dB** | **8.5442 dB** |
| Recovery MSE | 9770.2979 | 9091.9538 |
| Recovery MAE | 87.5373 | 82.6496 |
| Spatial predicted mask and IoU | N/A | N/A |

Recovery for both attacks took 1.345 s in total. DiffuEraser has the higher recovery PSNR by 0.3125 dB for this sample. This comparison concerns watermark recovery, not the visual quality of the inpainted video.


In [ ]:
show_videos([('Safe-Sora sampled original', 'media/02b72b8d/methods/safe-sora/sampled-original.mp4', 'Aligned 16-frame source used by Safe-Sora.'), ('Safe-Sora aligned mask', 'media/02b72b8d/methods/safe-sora/gt-mask.mp4', 'Mask sampled and resized to match the 16 input frames.'), ('Safe-Sora watermarked', 'media/02b72b8d/methods/safe-sora/watermarked.mp4', 'Graphical-watermarked video before object removal.'), ('Safe-Sora with ProPainter', 'media/02b72b8d/methods/safe-sora/propainter.mp4', 'ProPainter output after Safe-Sora embedding.'), ('Safe-Sora with DiffuEraser', 'media/02b72b8d/methods/safe-sora/diffueraser.mp4', 'DiffuEraser output after Safe-Sora embedding. Temporary padding is not shown.')])


### Recovered graphical watermarks

Each image shows the reference watermark on the left, the recovered watermark in the middle, and amplified pixel error on the right. These are not object masks.


In [ ]:
show_images([('Recovery after ProPainter', 'media/02b72b8d/methods/safe-sora/recovery-propainter.png', 'Recovery PSNR: 8.2317 dB.'), ('Recovery after DiffuEraser', 'media/02b72b8d/methods/safe-sora/recovery-diffueraser.png', 'Recovery PSNR: 8.5442 dB.')])


### Run records

The next cell prints the embedding, removal, and recovery JSON records, including sampled frame indices, compatibility details, fidelity metrics, and file fingerprints.


In [ ]:
show_metadata(['media/02b72b8d/methods/safe-sora/diffueraser.json', 'media/02b72b8d/methods/safe-sora/embed.json', 'media/02b72b8d/methods/safe-sora/propainter.json', 'media/02b72b8d/methods/safe-sora/recovery.json'])


## 2. Sample `0ab75563`


### Source and unwatermarked removal reference

The source has 104 frames at 960 x 448, 25 FPS, and a duration of about 4.16 seconds. The raw mask marks the object selected for removal. The two unwatermarked outputs show how ProPainter and DiffuEraser behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0ab75563/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0ab75563/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0ab75563/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0ab75563/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0ab75563/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0ab75563/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0ab75563/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0ab75563/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.')])


### Safe-Sora results

Safe-Sora selected 16 frames at indices `0, 7, 14, 21, 27, 34, 41, 48, 55, 62, 69, 76, 82, 89, 96, 103` and resized them to 512 x 320. Embedding took 5.677 s.

DiffuEraser requires at least 22 frames. The final frame and mask were repeated to 24 frames during inference, then the output was trimmed back to 16 genuine frames.

#### Embedding fidelity

| Comparison | PSNR | MSE | MAE |
| --- | ---: | ---: | ---: |
| Selected input to VAE cover | 35.6311 dB | 17.7816 | 2.9008 |
| VAE cover to watermarked output | **43.3406 dB** | 3.0132 | 1.2898 |

The first row measures ordinary VAE reconstruction change. The second row isolates the additional visible change caused by watermark embedding.

#### Removal and recovery

| Metric | ProPainter | DiffuEraser |
| --- | ---: | ---: |
| Removal runtime | 8.323 s | 27.844 s |
| Reported output frames | 16 | 16 |
| Recovery PSNR | **7.8723 dB** | **8.3995 dB** |
| Recovery MSE | 10613.2044 | 9400.0013 |
| Recovery MAE | 92.5715 | 86.0763 |
| Spatial predicted mask and IoU | N/A | N/A |

Recovery for both attacks took 1.249 s in total. DiffuEraser has the higher recovery PSNR by 0.5272 dB for this sample. This comparison concerns watermark recovery, not the visual quality of the inpainted video.


In [ ]:
show_videos([('Safe-Sora sampled original', 'media/0ab75563/methods/safe-sora/sampled-original.mp4', 'Aligned 16-frame source used by Safe-Sora.'), ('Safe-Sora aligned mask', 'media/0ab75563/methods/safe-sora/gt-mask.mp4', 'Mask sampled and resized to match the 16 input frames.'), ('Safe-Sora watermarked', 'media/0ab75563/methods/safe-sora/watermarked.mp4', 'Graphical-watermarked video before object removal.'), ('Safe-Sora with ProPainter', 'media/0ab75563/methods/safe-sora/propainter.mp4', 'ProPainter output after Safe-Sora embedding.'), ('Safe-Sora with DiffuEraser', 'media/0ab75563/methods/safe-sora/diffueraser.mp4', 'DiffuEraser output after Safe-Sora embedding. Temporary padding is not shown.')])


### Recovered graphical watermarks

Each image shows the reference watermark on the left, the recovered watermark in the middle, and amplified pixel error on the right. These are not object masks.


In [ ]:
show_images([('Recovery after ProPainter', 'media/0ab75563/methods/safe-sora/recovery-propainter.png', 'Recovery PSNR: 7.8723 dB.'), ('Recovery after DiffuEraser', 'media/0ab75563/methods/safe-sora/recovery-diffueraser.png', 'Recovery PSNR: 8.3995 dB.')])


### Run records

The next cell prints the embedding, removal, and recovery JSON records, including sampled frame indices, compatibility details, fidelity metrics, and file fingerprints.


In [ ]:
show_metadata(['media/0ab75563/methods/safe-sora/diffueraser.json', 'media/0ab75563/methods/safe-sora/embed.json', 'media/0ab75563/methods/safe-sora/propainter.json', 'media/0ab75563/methods/safe-sora/recovery.json'])


## 3. Sample `0b02886a`


### Source and unwatermarked removal reference

The source has 30 frames at 960 x 640, 25 FPS, and a duration of about 1.20 seconds. The raw mask marks the object selected for removal. The two unwatermarked outputs show how ProPainter and DiffuEraser behave before watermarking is introduced.

Direct links: [original](../../evidence/mose-task1/review50/media/0b02886a/original.mp4) | [raw mask](../../evidence/mose-task1/review50/media/0b02886a/gt_mask.mp4) | [ProPainter reference](../../evidence/mose-task1/review50/media/0b02886a/propainter.mp4) | [DiffuEraser reference](../../evidence/mose-task1/review50/media/0b02886a/diffueraser.mp4)


In [ ]:
show_videos([('Original MOSE video', 'media/0b02886a/original.mp4', 'Source video before watermarking or removal.'), ('Raw selected-object mask', 'media/0b02886a/gt_mask.mp4', 'White pixels identify the object selected for removal.'), ('ProPainter without watermark', 'media/0b02886a/propainter.mp4', 'Unwatermarked ProPainter removal reference.'), ('DiffuEraser without watermark', 'media/0b02886a/diffueraser.mp4', 'Unwatermarked DiffuEraser removal reference.')])


### Safe-Sora results

Safe-Sora selected 16 frames at indices `0, 2, 4, 6, 8, 10, 12, 14, 15, 17, 19, 21, 23, 25, 27, 29` and resized them to 512 x 320. Embedding took 4.960 s.

DiffuEraser requires at least 22 frames. The final frame and mask were repeated to 24 frames during inference, then the output was trimmed back to 16 genuine frames.

#### Embedding fidelity

| Comparison | PSNR | MSE | MAE |
| --- | ---: | ---: | ---: |
| Selected input to VAE cover | 35.2872 dB | 19.2469 | 2.8088 |
| VAE cover to watermarked output | **42.9434 dB** | 3.3018 | 1.3598 |

The first row measures ordinary VAE reconstruction change. The second row isolates the additional visible change caused by watermark embedding.

#### Removal and recovery

| Metric | ProPainter | DiffuEraser |
| --- | ---: | ---: |
| Removal runtime | 8.274 s | 28.032 s |
| Reported output frames | 16 | 16 |
| Recovery PSNR | **8.3260 dB** | **8.4703 dB** |
| Recovery MSE | 9560.4656 | 9248.0939 |
| Recovery MAE | 85.3032 | 82.4307 |
| Spatial predicted mask and IoU | N/A | N/A |

Recovery for both attacks took 0.959 s in total. DiffuEraser has the higher recovery PSNR by 0.1443 dB for this sample. This comparison concerns watermark recovery, not the visual quality of the inpainted video.


In [ ]:
show_videos([('Safe-Sora sampled original', 'media/0b02886a/methods/safe-sora/sampled-original.mp4', 'Aligned 16-frame source used by Safe-Sora.'), ('Safe-Sora aligned mask', 'media/0b02886a/methods/safe-sora/gt-mask.mp4', 'Mask sampled and resized to match the 16 input frames.'), ('Safe-Sora watermarked', 'media/0b02886a/methods/safe-sora/watermarked.mp4', 'Graphical-watermarked video before object removal.'), ('Safe-Sora with ProPainter', 'media/0b02886a/methods/safe-sora/propainter.mp4', 'ProPainter output after Safe-Sora embedding.'), ('Safe-Sora with DiffuEraser', 'media/0b02886a/methods/safe-sora/diffueraser.mp4', 'DiffuEraser output after Safe-Sora embedding. Temporary padding is not shown.')])


### Recovered graphical watermarks

Each image shows the reference watermark on the left, the recovered watermark in the middle, and amplified pixel error on the right. These are not object masks.


In [ ]:
show_images([('Recovery after ProPainter', 'media/0b02886a/methods/safe-sora/recovery-propainter.png', 'Recovery PSNR: 8.3260 dB.'), ('Recovery after DiffuEraser', 'media/0b02886a/methods/safe-sora/recovery-diffueraser.png', 'Recovery PSNR: 8.4703 dB.')])


### Run records

The next cell prints the embedding, removal, and recovery JSON records, including sampled frame indices, compatibility details, fidelity metrics, and file fingerprints.


In [ ]:
show_metadata(['media/0b02886a/methods/safe-sora/diffueraser.json', 'media/0b02886a/methods/safe-sora/embed.json', 'media/0b02886a/methods/safe-sora/propainter.json', 'media/0b02886a/methods/safe-sora/recovery.json'])


## Validation and provenance

- The committed validator decoded the videos, checked frame counts, dimensions, and FPS, and verified the required method fields.
- Aggregate and per-file validation: [validation.json](../../evidence/mose-task1/review50/validation.json)
- Aggregate results: [watermark-summary.json](../../evidence/mose-task1/review50/watermark-summary.json)
- Experimental contract: [PROTOCOL.md](../../evidence/mose-task1/PROTOCOL.md)
- Batch summary: [README.md](../../evidence/mose-task1/README.md)

The displayed values come from the committed Safe-Sora evidence records.
